In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
import warnings
import os

# Tắt cảnh báo chia cho 0 của scipy (do có những chu kỳ mạch nghỉ, activity = 0)
warnings.filterwarnings('ignore')

# ==========================================
# 1. TỰ ĐỘNG TÌM ĐƯỜNG DẪN FILE TRÊN KAGGLE
# ==========================================
print("Đang tự động quét tìm file trong Kaggle...")
VCD_FILE_PATH = None
LABELS_FILE_PATH = None

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file == 'mac_activity.vcd':
            VCD_FILE_PATH = os.path.join(root, file)
        elif file == 'tvla_labels.txt':
            LABELS_FILE_PATH = os.path.join(root, file)

if not VCD_FILE_PATH or not LABELS_FILE_PATH:
    print("❌ LỖI KHÔNG TÌM THẤY FILE!")
    print("Bạn vui lòng kiểm tra lại góc phải màn hình Kaggle (mục Data -> Input) xem file đã thực sự được tải lên chưa nhé.")
    # Dừng chương trình nếu không thấy
    raise FileNotFoundError("Thiếu file dữ liệu VCD hoặc Labels.")
else:
    print(f"✅ Đã tìm thấy file VCD tại: {VCD_FILE_PATH}")
    print(f"✅ Đã tìm thấy file Labels tại: {LABELS_FILE_PATH}")

CLOCK_PERIOD_PS = 10000  # 1 chu kỳ clock = 10ns = 10,000 ps
RESET_CYCLES = 25        # Bỏ qua 25 chu kỳ đầu tiên (250ns) lúc reset mạch

print("Đang tải nhãn (Labels)...")
labels = np.loadtxt(LABELS_FILE_PATH, dtype=int)
NUM_TRACES = len(labels)

# ==========================================
# 2. ĐỌC FILE VCD VÀ TÍNH SWITCHING ACTIVITY
# ==========================================
print("Đang phân tích file VCD... (Vui lòng chờ, Kaggle xử lý rất nhanh)")
cycle_activity = {}
current_time = 0

# Đọc từng dòng để tối ưu RAM 
with open(VCD_FILE_PATH, 'r') as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('$'):
            continue  # Bỏ qua các dòng khai báo header
            
        if line.startswith('#'):
            current_time = int(line[1:]) # Cập nhật thời gian hiện tại
        else:
            # Nếu không phải thời gian, đây là dòng tín hiệu bị lật (toggle)
            cycle_idx = current_time // CLOCK_PERIOD_PS
            cycle_activity[cycle_idx] = cycle_activity.get(cycle_idx, 0) + 1

# Chuyển dictionary thành mảng Numpy liên tục
max_cycle = max(cycle_activity.keys()) if cycle_activity else 0
activity_array = np.array([cycle_activity.get(i, 0) for i in range(max_cycle + 1)])

# ==========================================
# 3. CĂN CHỈNH VÀ CHIA TRACES
# ==========================================
print("Đang căn chỉnh Traces...")
# Cắt bỏ phần reset
activity_core = activity_array[RESET_CYCLES:]

# Tự động tính toán số chu kỳ clock cho mỗi trace
CYCLES_PER_TRACE = len(activity_core) // NUM_TRACES
print(f"-> Đã phát hiện {CYCLES_PER_TRACE} chu kỳ xung nhịp cho mỗi trace.")

# Ép mảng cho vừa vặn kích thước
activity_core = activity_core[:NUM_TRACES * CYCLES_PER_TRACE]

# Reshape thành ma trận 2D: [Số trace, Độ dài 1 trace]
trace_matrix = activity_core.reshape((NUM_TRACES, CYCLES_PER_TRACE))

# Chia làm 2 nhóm theo Labels
fixed_traces = trace_matrix[labels == 0]
random_traces = trace_matrix[labels == 1]

print(f"-> Nhóm Fixed: {len(fixed_traces)} traces")
print(f"-> Nhóm Random: {len(random_traces)} traces")

# ==========================================
# 4. TÍNH TOÁN TVLA T-VALUE (WELCH'S T-TEST)
# ==========================================
print("Đang tính toán T-value...")
t_values, p_values = ttest_ind(fixed_traces, random_traces, equal_var=False, axis=0)
t_values = np.nan_to_num(t_values, nan=0.0) # Khử nhiễu các điểm chia 0

# ==========================================
# 5. VẼ ĐỒ THỊ BÁO CÁO (YÊU CẦU MỤC 5)
# ==========================================
time_axis = np.arange(CYCLES_PER_TRACE)

plt.figure(figsize=(15, 12))

# --- ĐỒ THỊ 1: 10 Trace activity mẫu ---
plt.subplot(3, 1, 1)
for i in range(min(10, NUM_TRACES)):
    # Xác định trace này thuộc nhóm nào dựa vào mảng labels
    group_name = "Random" if labels[i] == 1 else "Fixed"
    plt.plot(time_axis, trace_matrix[i], alpha=0.8, label=f"Trace {i+1} ({group_name})")

plt.title("Đồ thị 1: Switching Activity của 10 Traces đầu tiên (Mục 5.1)", fontsize=12, fontweight='bold')
plt.ylabel("Số lượng Toggles (Activity)")
plt.grid(True, linestyle='--', alpha=0.6)

# MẸO XỬ LÝ LỖI ĐÈ HÌNH: Tự động tìm đỉnh cao nhất và nâng trần trục Y lên thêm 35%
y_max = np.max(trace_matrix[:min(10, NUM_TRACES)])
plt.ylim(0, y_max * 1.35) 

# Kích hoạt hiển thị bảng chú thích ở góc trên bên phải
plt.legend(loc='upper right', fontsize=9, ncol=5)

# --- ĐỒ THỊ 2: Trung bình hai nhóm ---
plt.subplot(3, 1, 2)
if len(fixed_traces) > 0 and len(random_traces) > 0:
    plt.plot(time_axis, np.mean(fixed_traces, axis=0), label="Mean of Fixed Group", color='blue', linewidth=2)
    plt.plot(time_axis, np.mean(random_traces, axis=0), label="Mean of Random Group", color='orange', linewidth=2)
plt.title("Đồ thị 2: So sánh trung bình Activity giữa nhóm Fixed và Random (Mục 5.2)", fontsize=12, fontweight='bold')
plt.ylabel("Mean Toggles")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# --- ĐỒ THỊ 3: TVLA t-value ---
plt.subplot(3, 1, 3)
plt.plot(time_axis, t_values, color='black', linewidth=1.5, label='t-value')
# Vẽ hai đường ngưỡng rò rỉ +/- 4.5
plt.axhline(4.5, color='red', linestyle='dashed', linewidth=2, label='Threshold (+4.5)')
plt.axhline(-4.5, color='red', linestyle='dashed', linewidth=2, label='Threshold (-4.5)')
plt.fill_between(time_axis, t_values, 4.5, where=(t_values > 4.5), color='red', alpha=0.3)
plt.fill_between(time_axis, t_values, -4.5, where=(t_values < -4.5), color='red', alpha=0.3)

plt.title("Đồ thị 3: Kết quả phân tích TVLA (t-value) theo thời gian (Mục 5.3)", fontsize=12, fontweight='bold')
plt.xlabel("Chu kỳ thời gian trong 1 Trace (Clock Cycles)")
plt.ylabel("t-value")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 6. PHÂN TÍCH CHI TIẾT SỐ LIỆU ĐỒ THỊ 1 & 2
# ==========================================
print("\n" + "="*75)
print(" 📊 BẢNG SỐ LIỆU TRÍCH XUẤT CHO BÁO CÁO (MỤC 6.3 - PHÂN TÍCH CHI TIẾT)")
print("="*75)

# --- PHÂN TÍCH ĐỒ THỊ 1: CHI TIẾT 10 TRACES ---
print("\n[1] ĐỒ THỊ 1 - CHI TIẾT 10 TRACES ĐẦU TIÊN:")
for i in range(min(10, NUM_TRACES)):
    group_name = "Random" if labels[i] == 1 else "Fixed"
    peak_val = np.max(trace_matrix[i])
    peak_cycle = np.argmax(trace_matrix[i])
    print(f"   -> Trace {i+1} ({group_name}): Đạt đỉnh cao nhất {peak_val:.0f} toggles (tại chu kỳ {peak_cycle})")

# --- THUẬT TOÁN TÌM "VÙNG ĐỒI" ---
# Tính trung bình toàn bộ traces để tìm ra vùng hoạt động cốt lõi
mean_all = np.mean(trace_matrix, axis=0)
# Đặt ngưỡng 20% so với đỉnh cao nhất để xác định lúc mạch bắt đầu chạy
threshold = np.max(mean_all) * 0.2  
active_cycles = np.where(mean_all > threshold)[0]
start_cycle = active_cycles[0]
end_cycle = active_cycles[-1]

print("\n[2] XÁC ĐỊNH 'VÙNG ĐỒI' (PHA HOẠT ĐỘNG MẠNH NHẤT):")
print(f"   -> Activity tạo thành vùng đồi liên tục từ: CHU KỲ {start_cycle} đến CHU KỲ {end_cycle}")
print(f"   -> Tổng số chu kỳ hoạt động mạnh (Pha CALC): {end_cycle - start_cycle + 1} chu kỳ")

# --- PHÂN TÍCH ĐỒ THỊ 2: CHÊNH LỆCH TRUNG BÌNH ---
if len(fixed_traces) > 0 and len(random_traces) > 0:
    print("\n[3] ĐỒ THỊ 2 - SO SÁNH NHÓM FIXED VÀ RANDOM TRONG 'VÙNG ĐỒI':")
    mean_fixed = np.mean(fixed_traces, axis=0)
    mean_random = np.mean(random_traces, axis=0)

    # Tính đỉnh trung bình của mỗi nhóm
    peak_fixed = np.max(mean_fixed)
    peak_random = np.max(mean_random)
    
    # Tính giá trị dao động trung bình CHỈ TRONG vùng đồi
    avg_hill_fixed = np.mean(mean_fixed[start_cycle:end_cycle+1])
    avg_hill_random = np.mean(mean_random[start_cycle:end_cycle+1])
    
    print(f"   -> Đỉnh trung bình cao nhất: Nhóm Fixed = {peak_fixed:.1f} | Nhóm Random = {peak_random:.1f}")
    print(f"   -> Mức tiêu thụ toggle trung bình (từ chu kỳ {start_cycle} đến {end_cycle}):")
    print(f"      + Nhóm Fixed:  ~{avg_hill_fixed:.1f} toggles/chu kỳ")
    print(f"      + Nhóm Random: ~{avg_hill_random:.1f} toggles/chu kỳ")
    print(f"   => KẾT LUẬN: Nhóm Random tiêu thụ năng lượng lớn hơn nhóm Fixed trung bình {avg_hill_random - avg_hill_fixed:.1f} toggles/chu kỳ.")
print("="*75 + "\n")

In [ ]:
# ==========================================
# 7. TRÍCH XUẤT SỐ LIỆU CHO ĐỒ THỊ 3 (TVLA)
# ==========================================
print("\n" + "="*75)
print(" 🛑 BẢNG SỐ LIỆU PHÂN TÍCH TVLA (MỤC 6.4)")
print("="*75)

# Tìm đỉnh t-value (cả chiều dương và chiều âm)
max_t_val = np.max(t_values)
min_t_val = np.min(t_values)
max_t_cycle = np.argmax(t_values)
min_t_cycle = np.argmin(t_values)

# Tìm đỉnh tuyệt đối (Peak |t|)
peak_abs_t = max(abs(max_t_val), abs(min_t_val))
peak_abs_cycle = max_t_cycle if abs(max_t_val) > abs(min_t_val) else min_t_cycle

# Tìm các chu kỳ bị rò rỉ (vượt ngưỡng +/- 4.5)
leaking_cycles = np.where((t_values > 4.5) | (t_values < -4.5))[0]

print(f"[1] KIỂM TRA NGƯỠNG RÒ RỈ (+/- 4.5):")
if len(leaking_cycles) > 0:
    print(f"   -> ⚠️ NGUY HIỂM: Thiết kế ĐÃ BỊ RÒ RỈ thông tin!")
    print(f"   -> Tổng số chu kỳ rò rỉ: {len(leaking_cycles)} chu kỳ")
    print(f"   -> Danh sách các chu kỳ rò rỉ: {leaking_cycles}")
else:
    print(f"   -> ✅ AN TOÀN: Không có chu kỳ nào vượt ngưỡng +/- 4.5.")

print(f"\n[2] ĐỈNH RÒ RỈ CAO NHẤT (Peak |t|):")
print(f"   -> Peak |t| = {peak_abs_t:.2f} (Xảy ra tại chu kỳ thứ {peak_abs_cycle})")

# Liên hệ với pha hoạt động
if len(leaking_cycles) > 0:
    # Lấy chu kỳ rò rỉ nằm trong khoảng CALC (0 đến 22)
    calc_leaks = [c for c in leaking_cycles if 0 <= c <= 22]
    print(f"\n[3] LIÊN HỆ MODULE / PHA HOẠT ĐỘNG:")
    print(f"   -> Có {len(calc_leaks)} điểm rò rỉ nằm trong pha CALC (tính toán cốt lõi).")
print("="*75 + "\n")